# ANALIZA PODATKOV O KNJIGAH

Analiza temelji na podatkih s spletne strani dobreknjige.si, in sicer na naboru knjig, ki jih ima v svoji zbirki Mestna knjižnica Ljubljana. Podatki vsebujejo naslov, avtorja, oceno, število ocen, obseg (število strani), oceno časa branja in podatek o tem, ali je bila knjiga kdaj nagrajena.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

pd.set_option("display.max_rows", 20)

%matplotlib inline

KNJIGE = pd.read_csv("podatki/knjige.csv", index_col="id")
KNJIGE

## Priprava podatkov

`cas_branja` je v podatkih zapisan kot besedilni razpon (npr. "8-9 ur"), zato ga pretvorimo v številsko povprečje razpona, da ga lahko uporabimo za izračune in grafe.

Prav tako velja opozoriti: `ocena` je pri knjigah, ki še nimajo nobene ocene (`stevilo_ocen == 0`), zapisana kot 0.0 - to NI dejanska ocena "0", ampak pomeni "še ni ocenjeno". Za vse izračune povprečnih ocen zato uporabljamo samo podmnožico knjig, ki imajo vsaj eno oceno.

In [ ]:
def povprecni_cas(vrednost):
    """Pretvori niz oblike '8-9 ur' ali '1 ur' v številsko povprečje (v urah)."""
    if pd.isna(vrednost) or vrednost == "":
        return None
    stevilke = [int(s) for s in re.findall(r"\d+", str(vrednost))]
    return sum(stevilke) / len(stevilke) if stevilke else None

KNJIGE["cas_branja_ur"] = KNJIGE["cas_branja"].apply(povprecni_cas)

# Podmnozica knjig, ki imajo vsaj eno oceno - uporabljena za vse analize ocen
OCENJENE = KNJIGE[KNJIGE["stevilo_ocen"] > 0]

print(f"Skupno knjig: {len(KNJIGE)}")
print(f"Knjig z vsaj eno oceno: {len(OCENJENE)}")

In [ ]:
manjkajoce_vrednosti = KNJIGE.isna().sum()
manjkajoce_vrednosti

## Analiza ocen

Porazdelitev ocen med vsemi ocenjenimi knjigami:

In [ ]:
ax = OCENJENE["ocena"].plot.hist(bins=20, color="cornflowerblue", edgecolor="black")
ax.set_xlabel("ocena")
ax.set_ylabel("stevilo knjig")
ax.set_title("Porazdelitev ocen knjig");

NAJBOLJE OCENJENE KNJIGE (upoštevamo samo tiste z vsaj 3 ocenami, da izločimo naključne visoke ocene z zelo malo glasovi):

In [ ]:
zanesljive_ocene = OCENJENE[OCENJENE["stevilo_ocen"] >= 3]

top10_najbolje = zanesljive_ocene[["naslov", "avtor", "ocena", "stevilo_ocen"]].sort_values(
    ["ocena", "stevilo_ocen"], ascending=[False, False]
).head(10)
top10_najbolje

NAJSLABŠE OCENJENE KNJIGE (enak pogoj, vsaj 3 ocene):

In [ ]:
top10_najslabse = zanesljive_ocene[["naslov", "avtor", "ocena", "stevilo_ocen"]].sort_values(
    ["ocena", "stevilo_ocen"], ascending=[True, False]
).head(10)
top10_najslabse

## Analiza obsega knjig

In [ ]:
ax = KNJIGE["stevilo_strani"].dropna().plot.hist(bins=30, color="mediumseagreen", edgecolor="black")
ax.set_xlabel("stevilo strani")
ax.set_ylabel("stevilo knjig")
ax.set_title("Porazdelitev dolzine knjig");

NAJDALJŠE KNJIGE:

In [ ]:
top10_najdaljse = KNJIGE[["naslov", "avtor", "stevilo_strani"]].sort_values("stevilo_strani", ascending=False).head(10)
top10_najdaljse

NAJKRAJŠE KNJIGE:

In [ ]:
top10_najkrajse = KNJIGE[["naslov", "avtor", "stevilo_strani"]].dropna(subset=["stevilo_strani"]).sort_values("stevilo_strani", ascending=True).head(10)
top10_najkrajse

## Nagrade

Delež knjig v zbirki, ki so bile kdaj nagrajene, in primerjava povprečne ocene med nagrajenimi in nenagrajenimi knjigami.

In [ ]:
delez_nagrajenih = KNJIGE["nagrajena"].mean() * 100
print(f"Delez nagrajenih knjig: {delez_nagrajenih:.1f} %")
print(f"Stevilo nagrajenih knjig: {KNJIGE['nagrajena'].sum()} od {len(KNJIGE)}")

In [ ]:
primerjava_nagrad = OCENJENE.groupby("nagrajena")["ocena"].agg(["mean", "count"]).round(2)
primerjava_nagrad

In [ ]:
ax = primerjava_nagrad["mean"].plot.bar(
    color=["indianred", "seagreen"],
    edgecolor="black",
    width=0.6
)
ax.set_xticklabels(["Nenagrajene", "Nagrajene"], rotation=0)
ax.set_ylabel("povprecna ocena")
ax.set_title("Povprecna ocena: nagrajene vs. nenagrajene knjige")

# priblizamo y-os na relevantno obmocje, da je majhna razlika sploh vidna
spodnja_meja = primerjava_nagrad["mean"].min() - 0.3
zgornja_meja = primerjava_nagrad["mean"].max() + 0.3
ax.set_ylim(spodnja_meja, zgornja_meja)

# dodamo tocne vrednosti nad vsak stolpec
for i, vrednost in enumerate(primerjava_nagrad["mean"]):
    ax.text(i, vrednost + 0.02, f"{vrednost:.2f}", ha="center", fontweight="bold");

## Avtorji

Avtorji z največ knjigami v zbirki Mestne knjižnice Ljubljana.

In [ ]:
top20_avtorjev = KNJIGE.groupby("avtor").size().sort_values(ascending=False).head(20)

ax = top20_avtorjev.plot.barh(color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("stevilo knjig v zbirki")
ax.set_title("20 avtorjev z najvec knjigami");

Povprečna ocena po avtorju (samo avtorji z vsaj 3 ocenjenimi knjigami v zbirki):

In [ ]:
ocena_po_avtorju = OCENJENE.groupby("avtor").agg(
    povprecna_ocena=("ocena", "mean"),
    stevilo_knjig=("naslov", "count")
)

zanesljivi_avtorji = ocena_po_avtorju[ocena_po_avtorju["stevilo_knjig"] >= 3]
zanesljivi_avtorji.sort_values("povprecna_ocena", ascending=False).round(2).head(15)

## Korelacije

Razmerja med številom strani, oceno, časom branja in številom ocen. Vsaka pika predstavlja eno knjigo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].scatter(KNJIGE["stevilo_strani"], KNJIGE["cas_branja_ur"], alpha=0.4)
axes[0].set_xlabel("stevilo strani")
axes[0].set_ylabel("ocenjeni cas branja [ur]")
axes[0].set_title("Stevilo strani vs. cas branja")

axes[1].scatter(OCENJENE["stevilo_ocen"], OCENJENE["ocena"], alpha=0.4, color="darkorange")
axes[1].set_xlabel("stevilo ocen")
axes[1].set_ylabel("ocena")
axes[1].set_title("Stevilo ocen vs. ocena")

axes[2].scatter(OCENJENE["stevilo_strani"], OCENJENE["ocena"], alpha=0.4, color="purple")
axes[2].set_xlabel("stevilo strani")
axes[2].set_ylabel("ocena")
axes[2].set_title("Stevilo strani vs. ocena");

In [ ]:
print("Korelacija stevilo_strani vs cas_branja_ur:", round(KNJIGE["stevilo_strani"].corr(KNJIGE["cas_branja_ur"]), 3))
print("Korelacija stevilo_ocen vs ocena:", round(OCENJENE["stevilo_ocen"].corr(OCENJENE["ocena"]), 3))
print("Korelacija stevilo_strani vs ocena:", round(OCENJENE["stevilo_strani"].corr(OCENJENE["ocena"]), 3))